In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, ClassLabel
import random

# =============================================================================
# 🛰️ PatternNet 데이터셋 탐험기: 우리 동네 지도를 읽는 AI!
#
# [데이터셋 제목] PatternNet
# [데이터셋 의미] 전 세계의 지표면(Land Cover) 이미지를 분류하는 데이터셋입니다.
# [설명] 위성사진을 기반으로 '여기는 공항이다', '여기는 해변이다', '여기는 숲이다'와 같은
#        지형지물을 분류하는 것이 목표입니다. 우주에서 찍은 사진을 보고 무엇인지 맞추는
#        굉장히 흥미진진한 AI 실습입니다!
# =============================================================================

# --- 설정 변수 ---
DATASET_NAME = "blanchon/PatternNet"
SPLIT_NAME = "train"
SAMPLE_COUNT = 5  # 초보자 실습을 위해 샘플링 할 개수를 5개로 설정합니다.

# -----------------------------------------------------------------------------
# 💡 튜터 코멘트: 데이터 로드 전략 (Streaming vs. Local)
# 스트리밍 모드는 메모리를 효율적으로 사용하지만, 특정 기능을 제한할 수 있습니다.
# 따라서 먼저 스트리밍으로 시도하고, 실패하면 소량만 다운로드하는 방식을 사용합니다.
# -----------------------------------------------------------------------------

dataset = None
print("=====================================================")
print("✅ 1단계: 데이터셋 로드 준비 (Streaming Attempt)")
print("=====================================================")

try:
    # ⭐️ 먼저 스트리밍 방식으로 시도 (가장 빠르고 메모리 효율적)
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print(f"✨ 성공! 데이터셋 '{DATASET_NAME}'을 스트리밍(Streaming) 모드로 로드했습니다. (메모리 절약!)")
except Exception as e:
    print(f"⚠️ 경고: 스트리밍 로드 중 오류 발생 ({e}).")
    print("➡️ 일반 모드로 소량만 다운로드하여 작업을 진행합니다.")
    # 스트리밍 실패 시, 소량의 데이터를 일반 모드로 로드합니다.
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)


# -----------------------------------------------------------------------------
# 🧬 2단계: 데이터 구조 분석 및 클래스 이해하기
# (지구 표면에서 어떤 '단서'들을 찾을지 미리 파악하는 과정!)
# -----------------------------------------------------------------------------

print("\n\n=====================================================")
print("📚 2단계: 데이터 구조 파헤치기 (Labels 확인)")
print("=====================================================")

# 1. 전역 Config 정보 가져오기
try:
    from datasets import get_dataset_config_names
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"ℹ️ 사용 가능한 Config 목록: {configs}")
except Exception as e:
    print("ℹ️ Config 정보를 가져오지 못했습니다. 기본 설정으로 진행합니다.")


# 2. 레이블(Label) 정보를 확인하여 어떤 장소들이 있는지 살펴봅시다.
# PatternNet의 레이블은 실제로 데이터셋 메타 정보에 나와있습니다.
# 가장 자주 사용되는 Label 구조를 파악하는 것이 중요해요!
print("\n[🌎 이 데이터셋이 담고 있는 '장소'의 종류 (일부만 예시):]")
# ClassLabel을 활용하여 레이블 명칭을 뽑아냅니다.
try:
    # metadata를 통해 레이블을 추출합니다.
    label_info = dataset.features['label'].info.names
    print(f"총 {len(label_info)}개의 클래스가 존재합니다.")
    print("✨ 주요 클래스:", label_info[:5], "...")
    print("✨ 흥미로운 지형물 예시:", "beach(해변)", "forest(숲)", "freeway(고속도로)", "airport(비행장)", "river(강)")
except Exception as e:
    print(f"⚠️ 레이블 정보 접근 중 오류가 발생했습니다. {e}")


# -----------------------------------------------------------------------------
# ✨ 3단계: 창의적 실습 - '랜덤 지역 탐색' (Random Spot Analysis)
# -----------------------------------------------------------------------------

print("\n\n=====================================================")
print(f"🗺️ 3단계: {SAMPLE_COUNT}개의 '랜덤 지역' 탐색 및 분석")
print("=====================================================")

# 스트리밍 방식이든, 일반 방식이든, .take()를 이용해 상위 K개만 가져옵니다.
# 주의: len()을 사용하지 않으므로, 반복문(for)을 사용하는 것이 가장 안전합니다.
sample_data_iterator = dataset.take(SAMPLE_COUNT)
sample_data_list = []
for i, sample in enumerate(sample_data_iterator):
    sample_data_list.append(sample)

print(f"\n🔍 준비 완료! 총 {len(sample_data_list)}개의 샘플을 분석합니다.")

# -----------------------------------------------------------------------------
# 실습 목표:
# 1. 이미지의 기본 정보를 뽑아보고 (shape, size).
# 2. 레이블을 사람이 이해할 수 있는 이름으로 변환해봅니다.
# 3. 여러 샘플을 시각화하여 패턴을 발견해봅니다.
# -----------------------------------------------------------------------------

# 샘플들을 순회하며 분석을 진행합니다.
for i, sample in enumerate(sample_data_list):
    image_data = sample['image']
    label_index = sample['label']

    # 🌟 데이터 타입 처리: PIL 이미지 객체(PIL.Image.Image)인지 확인하고 처리합니다.
    # 이미지 배열로 변환하여 분석하기 쉽게 만듭니다.
    if hasattr(image_data, 'size'):
        # PIL Image 객체일 경우, numpy 배열로 변환 (가장 흔한 시나리오)
        try:
            image_np = np.array(image_data)
        except Exception:
            image_np = None
    else:
        # 이미 numpy 배열인 경우 (예외 처리)
        image_np = image_data

    # 레이블 이름 가져오기
    try:
        label_name = label_index.name
    except AttributeError:
        label_name = f"Unknown (Index: {label_index})"

    print(f"\n--- 🗺️ 샘플 #{i+1}: (Label: {label_name}) ---")
    print(f"   ✨ 분석 결과: 이 지역은 '{label_name}'일 가능성이 높습니다.")

    # --------------------------------------------------
    # 🚀 AI 시각 분석 (매직 파트!)
    # --------------------------------------------------
    if image_np is not None:
        print(f"   🖼️ 이미지 크기 (Shape): {image_np.shape}")
        print("   ✨ Mission Success! Numpy 배열로 성공적으로 변환되었습니다.")

        # 시각화를 위해 Matplotlib을 사용합니다.
        plt.figure(figsize=(6, 6))
        plt.imshow(image_np)
        plt.title(f"Land Cover: {label_name}", fontsize=14)
        plt.axis('off') # 축 눈금 숨기기
        plt.show()

    else:
        print("   ❌ 이미지 데이터 로드에 실패했습니다. (이 샘플은 건너뜁니다.)")


# -----------------------------------------------------------------------------
# 🌟 4단계: 창의적 결론 도출 - '희귀 클래스 분석'
# -----------------------------------------------------------------------------

print("\n\n=====================================================")
print("💡 4단계: 지표 학습 (Land Cover Insight)")
print("=====================================================")

print("🌟 [지표 AI 튜터의 인사이트]:")
print("PatternNet은 다양한 '인공 구조물(Man-made)'과 '자연 환경(Nature)'을 분류합니다.")
print("   - 자연물 예시: forest(숲), beach(해변), river(강)")
print("   - 인공물 예시: freeway(고속도로), airport(비행장), intersection(교차점)")
print("\n✅ 실습을 통해 여러분은 위성 이미지만 보고 '이것은 자연일까? 아니면 인간이 만든 시설일까?'라는 질문에 답하는 AI의 사고방식을 경험했습니다!")
print("\n🎉 축하합니다! 데이터셋의 구조를 파악하고, 실제 데이터의 내용을 읽어내는 훌륭한 경험을 했습니다!")